# CLT Forge: Validated PowerShell Circuit View

This notebook uses the CLT-Forge graph rendering utilities to build a project-specific
attribution-style visualization for the current validated claim in this repo.

It does **not** retrain CLTs. Instead, it uses CLT-Forge's graph visualization helper to
render a compact mechanistic summary of the validated route:

- minimal direct branch: `L0H11 -> L12H15/L12H5/L12H4`
- stronger sufficiency-oriented late carrier: `L12H15/L12H5/L12H4/L12H28`
- auxiliary grouped-ablation-sensitive helper: `L12H2`
- evasion interpretation: downstream redistribution, not simple route deletion


## What This Visualization Does And Does Not Claim

This notebook uses CLT-Forge only for **visual rendering**. The graph is a hand-constructed summary of the repo's validated results, not a newly computed CLT attribution graph.

That means the visualization should be read as a communication aid for the existing evidence in `FINDINGS.md`:

- It **does** summarize the current 96-pair claim hierarchy.
- It **does not** establish any new mechanistic result beyond the interventions already reported in the repo.
- It is best read as a compact map of the validated late route and its intervention logic.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/LLM-Interp/CLT-Forge.git"
LOCAL_REPO = Path("/tmp/CLT-Forge")

if not LOCAL_REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)

src_path = LOCAL_REPO / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Using CLT-Forge from:", LOCAL_REPO)


In [ ]:
from IPython.display import Image, SVG, display

from clt_forge.vendor.circuit_tracer.demos.graph_visualization import (
    Feature,
    InterventionGraph,
    Supernode,
    create_graph_visualization,
)


def assign_activation(node, value):
    node.activation = value
    return node


def feature(layer, pos, idx):
    return Feature(layer=layer, pos=pos, feature_idx=idx)


def build_validated_route_graph():
    indicator_tokens = assign_activation(
        Supernode("indicator tokens", [feature(0, 0, 1)]), 1.0
    )
    layer0_h11 = assign_activation(
        Supernode("L0H11", [feature(0, 1, 11)]), 0.91
    )
    late_core = assign_activation(
        Supernode("L12H15/H5/H4", [feature(12, 2, 15), feature(12, 2, 5), feature(12, 2, 4)]),
        0.88,
    )
    h28 = assign_activation(Supernode("L12H28", [feature(12, 2, 28)]), 0.72)
    h2 = assign_activation(Supernode("L12H2", [feature(12, 2, 2)]), 0.54)
    resid_pre13 = assign_activation(
        Supernode("resid_pre13\nmalicious direction", [feature(13, 2, 0)]), 0.84
    )
    block_logit = assign_activation(Supernode("BLOCK logit", [feature(31, 2, 0)]), 0.94)

    indicator_tokens.children = [layer0_h11]
    layer0_h11.children = [late_core]
    late_core.children = [resid_pre13]
    h28.children = [resid_pre13]
    h2.children = [resid_pre13]
    resid_pre13.children = [block_logit]

    ordered_nodes = [
        [indicator_tokens],
        [layer0_h11, late_core, h28, h2],
        [resid_pre13],
        [block_logit],
    ]
    prompt = "PowerShell sample with shared suspicious indicators; validated 96-pair route summary"
    graph = InterventionGraph(ordered_nodes=ordered_nodes, prompt=prompt)
    top_outputs = [("BLOCK", 0.82), ("ALLOW", 0.11), ("malicious", 0.04)]
    return graph, top_outputs


def build_ablation_graph():
    indicator_tokens = assign_activation(
        Supernode("indicator tokens", [feature(0, 0, 1)]), 1.0
    )
    layer0_h11 = assign_activation(
        Supernode("L0H11", [feature(0, 1, 11)]), 0.91
    )
    replacement = assign_activation(
        Supernode("weakened late carrier", [feature(12, 2, 99)]), 0.23
    )
    late_core = assign_activation(
        Supernode(
            "L12H15/H5/H4/H28",
            [feature(12, 2, 15), feature(12, 2, 5), feature(12, 2, 4), feature(12, 2, 28)],
            intervention="ablate",
            replacement_node=replacement,
        ),
        0.17,
    )
    h2 = assign_activation(
        Supernode("L12H2\naux helper", [feature(12, 2, 2)], intervention="keep"), 0.48
    )
    resid_pre13 = assign_activation(
        Supernode("resid_pre13\nweakened evidence", [feature(13, 2, 0)]), 0.29
    )
    allow_logit = assign_activation(Supernode("ALLOW logit", [feature(31, 2, 1)]), 0.63)

    indicator_tokens.children = [layer0_h11]
    layer0_h11.children = [late_core]
    late_core.children = [resid_pre13]
    replacement.children = [resid_pre13]
    h2.children = [resid_pre13]
    resid_pre13.children = [allow_logit]

    ordered_nodes = [
        [indicator_tokens],
        [layer0_h11, late_core, h2],
        [resid_pre13],
        [allow_logit],
    ]
    prompt = "Late-carrier ablation view: grouped intervention weakens malicious evidence and the final BLOCK preference"
    graph = InterventionGraph(ordered_nodes=ordered_nodes, prompt=prompt)
    top_outputs = [("ALLOW", 0.61), ("BLOCK", 0.27), ("benign", 0.06)]
    return graph, top_outputs


def build_github_style_claim_graph():
    indicator_tokens = assign_activation(
        Supernode("Shared suspicious\nindicator tokens", [feature(0, 0, 1)]), 0.96
    )
    layer0_h11 = assign_activation(
        Supernode("Clean upstream head\nL0H11", [feature(0, 1, 11)]), 0.9
    )
    minimal_branch = assign_activation(
        Supernode("Minimal direct branch\nL12H15/H5/H4", [feature(12, 2, 15), feature(12, 2, 5), feature(12, 2, 4)]), 0.87
    )
    h28 = assign_activation(
        Supernode("Stronger sufficiency helper\nL12H28", [feature(12, 2, 28)]), 0.7
    )
    h2 = assign_activation(
        Supernode("Auxiliary family-sensitive\nhelper L12H2", [feature(12, 2, 2)]), 0.52
    )
    resid_pre13 = assign_activation(
        Supernode("Late malicious-evidence\nwrite at resid_pre13", [feature(13, 2, 0)]), 0.83
    )
    block_logit = assign_activation(
        Supernode("Final BLOCK preference", [feature(31, 2, 0)]), 0.93
    )

    indicator_tokens.children = [layer0_h11]
    layer0_h11.children = [minimal_branch]
    minimal_branch.children = [resid_pre13]
    h28.children = [resid_pre13]
    h2.children = [resid_pre13]
    resid_pre13.children = [block_logit]

    ordered_nodes = [
        [indicator_tokens],
        [layer0_h11],
        [minimal_branch, h28, h2],
        [resid_pre13],
        [block_logit],
    ]
    prompt = "GitHub-style CLT-Forge route figure: validated 96-pair claim hierarchy"
    graph = InterventionGraph(ordered_nodes=ordered_nodes, prompt=prompt)
    top_outputs = [("BLOCK", 0.82), ("ALLOW", 0.11), ("redistribution under evasion", 0.05)]
    return graph, top_outputs


## How To Read The First Figure

The first figure is the **claim summary** view. It is intended to help a junior researcher visually separate three related but different statements:

- `L0H11` is the cleanest validated upstream entry point.
- `L12H15/L12H5/L12H4` is the minimal direct late route.
- Adding `L12H28` gives the stronger sufficiency-oriented late carrier.

Why is `L12H2` drawn separately? Because the repo's results say it is not part of the most stable sufficiency core. Instead, it behaves more like an auxiliary helper that matters more under some grouped ablations than under the main path-patching comparison.

The displayed activation percentages are illustrative labels for the visualization. The actual evidence comes from the intervention metrics in the findings, especially the 96-pair `mean Δ` and `flip_rate` comparisons.


In [ ]:
validated_graph, validated_outputs = build_validated_route_graph()
validated_svg = create_graph_visualization(validated_graph, validated_outputs)
display(validated_svg)


The first view is a compact attribution-style summary of the main repo claim.

- `L0H11` is shown as the clean upstream entry point.
- `L12H15/H5/H4` is the minimal late route.
- `L12H28` strengthens the sufficiency-oriented late carrier.
- `L12H2` is shown separately because the findings treat it as auxiliary rather than part of the stable sufficiency core.


## How To Read The Second Figure

The second figure is the **intervention interpretation** view. It visualizes the idea behind grouped ablation of the validated late carrier:

- ablating the late carrier weakens the malicious-evidence write
- the downstream decision becomes less `BLOCK`-leaning
- this is why grouped ablation is treated as a **necessity** test in the writeup

This figure should not be read as showing literal neuron-level replacement values from a CLT run. It is a conceptual rendering of the repo's claim that the late carrier is causally important, while `L12H2` is auxiliary and the whole late stage is only partially decomposed.

For this project, the most important conceptual distinction is:

- **path patching** asks whether the route is sufficient to move the decision
- **grouped ablation** asks whether removing the route weakens the decision


In [ ]:
ablation_graph, ablation_outputs = build_ablation_graph()
ablation_svg = create_graph_visualization(ablation_graph, ablation_outputs)
display(ablation_svg)


## GitHub-Style Example

The repo also uses a more presentation-oriented static figure style in its GitHub-facing artifacts.
This section shows one of those existing figures and then places the CLT-Forge rendering next to it as a concrete example of how the two styles relate.

How to read this comparison:

- the GitHub-style image is the polished communication view already used elsewhere in the repo
- the CLT-Forge SVG is the attribution-style counterpart built in this notebook
- both are summarizing the same validated late-route claim rather than introducing a new result


In [ ]:
artifacts_dir = Path("../artifacts").resolve()
github_style_path = artifacts_dir / "figure_validated_route.png"

print("GitHub-style figure:", github_style_path)
display(Image(filename=str(github_style_path), width=900))

print("CLT-Forge counterpart:")
display(SVG(data=validated_svg.data))


## Third Graph Attempt: More GitHub-Style Labeling

The next view is a second CLT-Forge attempt designed to feel closer to the static GitHub-page route figure.
The goal is not to copy the SVG exactly. It is to shift the node labels and layout toward a more presentation-oriented style:

- explicit role labels such as "clean upstream head" and "final BLOCK preference"
- a clearer separation between the minimal direct branch and the two helper heads
- a figure that reads more like a polished claim summary than an internal mechanistic sketch


In [ ]:
github_claim_graph, github_claim_outputs = build_github_style_claim_graph()
github_claim_svg = create_graph_visualization(github_claim_graph, github_claim_outputs)
display(github_claim_svg)


## Output Files

The final cell writes both SVGs to `../artifacts/` so they can be reused in docs or slides.
The GitHub-style PNG is an existing artifact and is displayed above for comparison rather than regenerated here.

- `clt_forge_validated_route.svg`: high-level validated-route summary
- `clt_forge_ablation_view.svg`: intervention-oriented late-carrier view
- `clt_forge_github_style_claim.svg`: more presentation-oriented CLT-Forge route figure


In [ ]:
artifacts_dir = Path("../artifacts").resolve()
artifacts_dir.mkdir(parents=True, exist_ok=True)

validated_path = artifacts_dir / "clt_forge_validated_route.svg"
ablation_path = artifacts_dir / "clt_forge_ablation_view.svg"
github_claim_path = artifacts_dir / "clt_forge_github_style_claim.svg"

validated_path.write_text(validated_svg.data, encoding="utf-8")
ablation_path.write_text(ablation_svg.data, encoding="utf-8")
github_claim_path.write_text(github_claim_svg.data, encoding="utf-8")

print("Saved:")
print(validated_path)
print(ablation_path)
print(github_claim_path)
